In [62]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import pickle

In [31]:
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    GlobalMaxPooling1D,
    Dropout,
    Lambda
)

from tensorflow.keras.callbacks import EarlyStopping

In [11]:
df=pd.read_csv("siamese_metadata_dataset.csv")

df=df[
    df["input_a"]!=df["input_b"]
]

df.head()

,input_a,input_b,label
0,Brown Whitehorse Boots Shoes For Men,Brown Whitehorse Boots Shoe's For Men,1
1,Goldstar Boost 12 White Maroon Goldstar Shoes ...,Goldstarr Boost 12 White Maroon Goldstar Shoe'...,1
2,ERKE Cushioning Running Shoes Black/White For ...,Premium 1:1 Quality ERKI Cushioning Runin Shoe...,1
3,Adidas GALAXY STAR M Shoes For Men (IG0761) BY...,Men's LPMX casual shoes- 912411301-BLK,0
4,PEAK Cushion Running Shoes White For Men E224007H,Vomeros Premium Originals Sports Running/Train...,0


In [12]:
print(df.shape)
print(df["label"].value_counts())
df.isnull().sum()

(4080, 3)
label
0    2041
1    2039
Name: count, dtype: int64


input_a    0
input_b    0
label      0
dtype: int64

In [13]:
X1=df["input_a"]
X2=df["input_b"]
y=df["label"]

(
    X1_train,
    X1_test,

    X2_train,
    X2_test,

    y_train,
    y_test
) = train_test_split(
    X1, X2, y,

    test_size=0.2,
    random_state=42
)

In [14]:
tokenizer=Tokenizer(
    num_words=15000,
    oov_token="<OOV>"
)

In [15]:
all_text=pd.concat([
    X1_train,
    X2_train
])

tokenizer.fit_on_texts(all_text)

In [16]:
X1_train_seq=tokenizer.texts_to_sequences(X1_train)
X1_test_seq=tokenizer.texts_to_sequences(X1_test)

X2_train_seq=tokenizer.texts_to_sequences(X2_train)
X2_test_seq=tokenizer.texts_to_sequences(X2_test)

In [17]:
lengths=[len(seq) for seq in X1_train_seq]

print("Average Length:", np.mean(lengths))
print("Max Length:", np.max(lengths))

Average Length: 9.462316176470589
Max Length: 24


In [18]:
max_length=24

X1_train_pad=pad_sequences(
    X1_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X1_test_pad=pad_sequences(
    X1_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X2_train_pad=pad_sequences(
    X2_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X2_test_pad=pad_sequences(
    X2_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [19]:
vocab_size=len(tokenizer.word_index)+1
print(vocab_size)

1662


In [52]:
input_layer=Input(shape=(max_length,))

embedding=Embedding(
    input_dim=vocab_size,
    output_dim=128
)(input_layer)

bilstm=Bidirectional(
    LSTM(64, return_sequences=True)           #Pooling layers needs full sequence outputs.
)(embedding)

pooled=GlobalMaxPooling1D()(bilstm)           #This compresses: (sequence_length, features) into strongest semantic signals.

embedding_vector=Dense(
    128,
    activation='relu',
    name='metadata_embedding'                 #this is the actual metadata embedding.
)(pooled)

In [53]:
encoder=Model(
    inputs=input_layer,
    outputs=embedding_vector                  #real reusable model
)

In [54]:
input_a=Input(shape=(max_length,))
input_b=Input(shape=(max_length,))

embedding_a=encoder(input_a)
embedding_b=encoder(input_b)

In [55]:
def cosine_similarity(vectors):

    x, y=vectors

    x=tf.math.l2_normalize(x, axis=1)
    y=tf.math.l2_normalize(y, axis=1)

    return tf.reduce_sum(
        x*y,
        axis=1,
        keepdims=True
    )

In [56]:
similarity=Lambda(
    cosine_similarity
)([embedding_a, embedding_b])

In [57]:
siamese_model=Model(
    inputs=[input_a, input_b],
    outputs=similarity
)

In [58]:
siamese_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [59]:
siamese_model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)   │ (None, 24)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_11 (InputLayer)   │ (None, 24)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ functional_6 (Functional)     │ (None, 128)               │         328,064 │ input_layer_10[0][0],      │
│                               │                           │                 │ input_layer_11[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lambda_4 (Lambda)             │ (None, 1)                 │               0 │ functional_6[0][0],        │
│                               │                           │                 │ functional_6[1][0]         │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 328,064 (1.25 MB)

 Trainable params: 328,064 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [60]:
early_stop=EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [61]:
history=siamese_model.fit(
    [X1_train_pad, X2_train_pad],
    y_train,

    validation_data=(
        [X1_test_pad, X2_test_pad],
        y_test
    ),

    epochs=10,
    batch_size=32,

    callbacks=[early_stop]
)

Epoch 1/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 0.0853 - mae: 0.2236 - val_loss: 0.0537 - val_mae: 0.1619
Epoch 2/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0239 - mae: 0.0953 - val_loss: 0.0397 - val_mae: 0.1267
Epoch 3/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0097 - mae: 0.0542 - val_loss: 0.0511 - val_mae: 0.1520
Epoch 4/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0071 - mae: 0.0439 - val_loss: 0.0329 - val_mae: 0.1006
Epoch 5/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0035 - mae: 0.0293 - val_loss: 0.0295 - val_mae: 0.0860
Epoch 6/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0015 - mae: 0.0173 - val_loss: 0.0293 - val_mae: 0.0803
Epoch 7/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 8.2130e-04 - mae: 0.0129 - val_loss: 0.0292 - val_mae: 0.0780
Epoch 8/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 9.6487e-04 - mae: 0.0137 - val_loss: 0.0292 - val_mae: 0.0754
Epoch 9/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 

In [63]:
encoder.save(
    "metadata_embedding_encoder_v1.keras"
)

In [64]:
with open("metadata_tokenizer_v1.pkl", "wb") as f:
    pickle.dump(tokenizer, f)